# Cyclistic — Section 3: MapReduce with Spark Core (RDD)

Optimises the two slowest steps from Section 2 using **Spark Core RDD API only**.
No DataFrames, no Spark SQL.

**Candidate 1:** Popular routes — GROUP BY (start_station, end_station), COUNT  
**Candidate 2:** Busiest stations — union of departures + arrivals per station, SUM  

**Dataset:** `data/processed/trips_clean.csv` (~14.8 million rows, 2020–2022)

In [ ]:
%pip install pyspark==3.5.1 --quiet

In [ ]:
import time
import os
import sys
from datetime import datetime

os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

NB_START = time.time()
print("Python:", sys.executable)

In [ ]:
_t = time.time()

from pyspark.sql import SparkSession
from pyspark import SparkContext

# Stop any existing context so the notebook is safe to re-run
existing = SparkContext._active_spark_context
if existing:
    existing.stop()

spark = (
    SparkSession.builder
    .appName("CyclisticRDD")
    .master("local[*]")
    .config("spark.python.worker.reuse", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version:", spark.version)
print(f"Cell time: {time.time() - _t:.2f}s")

## Load & Enrich

Read raw text, skip header, parse timestamps, compute `duration_minutes`.
Each row is reduced to a 3-tuple `(start_station, end_station, duration_minutes)` — discarding unused fields cuts per-record memory roughly 3×.
Cache the cleaned RDD — both candidates read from it.

In [ ]:
_t = time.time()

abs_path  = os.path.abspath("data/processed/trips_clean.csv").replace("\\", "/")
DATA_PATH = f"file:///{abs_path}"

# Columns: 0 ride_id | 1 started_at | 2 ended_at | 3 start_station_name
#           4 start_station_id | 5 end_station_name | 6 end_station_id | 7 member_casual

def enrich(line):
    """Return (start_station, end_station, duration_minutes) or None."""
    try:
        r = line.strip().split(",")
        if len(r) != 8:
            return None
        t0 = datetime.fromisoformat(r[1])
        t1 = datetime.fromisoformat(r[2])
        duration = (t1 - t0).total_seconds() / 60
        # Only keep the three fields both candidates need
        return (r[3], r[5], duration)
    except Exception:
        return None

clean_rdd = (
    sc.textFile(DATA_PATH)
    .filter(lambda line: not line.startswith("ride_id"))
    .map(enrich)
    .filter(lambda r: r is not None and 0 < r[2] <= 1440)
    .cache()
)

total = clean_rdd.count()
print(f"Rows after filter: {total:,}")
print(f"Cell time: {time.time() - _t:.2f}s")

## Candidate 1 — Popular Routes

**Prototype step:** `df.groupby(["start_station_name", "end_station_name"]).size()` — slowest analytical step at **0.1993s** on 148k rows. At 14.8M rows the composite key space (~500k unique pairs) forces a full in-memory hash-table build on a single thread.

**Why MapReduce helps:** each partition independently hashes its own key subset (map); only partial counts are shuffled to reducers (reduce). The expensive aggregation is distributed rather than centralised.

**Expected speedup:** linear with number of cores (4 logical CPUs) → ~4× on aggregation.

| Phase | Operation |
|-------|-----------|
| **Map** | `line → ((start_station, end_station), 1)` |
| **Reduce** | `(key, [1, 1, …]) → (key, sum)` |

In [ ]:
_t = time.time()

# Map: emit (route_key, 1) for each valid row — r = (start_station, end_station, duration)
def map_route(r):
    return ((r[0], r[1]), 1)

popular_routes = (
    clean_rdd
    .filter(lambda r: r[0] and r[1])
    .map(map_route)                               # ((start, end), 1)
    .reduceByKey(lambda a, b: a + b)              # ((start, end), count)
    .collect()
)
popular_routes.sort(key=lambda x: -x[1])

spark_routes_s = time.time() - _t

print(f"Unique routes: {len(popular_routes):,}")
print(f"\nTop 10 routes by trip count:")
print(f"  {'start_station':<40} {'end_station':<40} {'trips':>6}")
print("  " + "-" * 90)
for (start, end), count in popular_routes[:10]:
    print(f"  {start:<40} {end:<40} {count:>6,}")

print(f"\nSpark RDD time  : {spark_routes_s:.2f}s  (~14.8M rows)")
print(f"Pandas baseline : 0.1993s  (148k rows, 100× smaller)")

## Candidate 2 — Busiest Stations (Combined Traffic)

**Prototype step:** `pd.concat([departures, arrivals]).groupby(station).sum()` — **0.0653s** on 148k rows, second slowest. Requires two independent full-table scans then a merge/reduce per station.

**Why MapReduce helps:** `flatMap` emits one event per departure *and* per arrival in a single pass. The reduce phase sums them — matching the canonical word-count MapReduce pattern. Single-threaded pandas must scan the column twice sequentially; Spark scans each partition once and combines.

**Expected speedup:** ~2–4× due to single-pass scan vs. two sequential scans, spread across 4 cores.

| Phase | Operation |
|-------|-----------|
| **Map (flatMap)** | `row → [(start_station, 1), (end_station, 1)]` |
| **Reduce** | `(station, [1, 1, …]) → (station, total_activity)` |

In [ ]:
_t = time.time()

# FlatMap: emit one (station, 1) for departure AND one for arrival — r = (start, end, duration)
def map_station_events(r):
    events = []
    if r[0]:
        events.append((r[0], 1))
    if r[1]:
        events.append((r[1], 1))
    return events

busiest_stations = (
    clean_rdd
    .flatMap(map_station_events)          # (station, 1) for each departure/arrival
    .reduceByKey(lambda a, b: a + b)      # (station, total_activity)
    .collect()
)
busiest_stations.sort(key=lambda x: -x[1])

spark_stations_s = time.time() - _t

print(f"Unique stations: {len(busiest_stations):,}")
print(f"\nTop 10 busiest stations by total activity:")
print(f"  {'station_name':<45} {'total_activity':>14}")
print("  " + "-" * 62)
for station, total in busiest_stations[:10]:
    print(f"  {station:<45} {total:>14,}")

print(f"\nSpark RDD time  : {spark_stations_s:.2f}s  (~14.8M rows)")
print(f"Pandas baseline : 0.0653s  (148k rows, 100× smaller)")

## Summary

In [ ]:
total_nb = time.time() - NB_START

print("=" * 60)
print("SECTION 3 — RESULTS SUMMARY")
print("=" * 60)
print(f"  {'Step':<38} {'Spark (14.8M)':>13} {'Pandas (148k)':>13}")
print("  " + "-" * 66)
print(f"  {'Candidate 1 — Popular routes':<38} {spark_routes_s:>12.2f}s {'0.1993s':>13}")
print(f"  {'Candidate 2 — Busiest stations':<38} {spark_stations_s:>12.2f}s {'0.0653s':>13}")
print("  " + "-" * 66)
print(f"  {'Total notebook wall time':<38} {total_nb:>12.2f}s")
print("=" * 60)
print()
print("Analysis:")
print("  Both candidates run on 100× more data than the prototype.")
print("  Spark's per-partition map phase keeps aggregation local before shuffle,")
print("  avoiding the single-node memory bottleneck seen in pandas and SQLite.")

sc.stop()